# Building and testing a synthetic multilevel linear model

This notebook develops a small model for testing the `MultilevelModel` interface. It is intentionally simple: every level has a known solution, but evaluating that level still requires building and solving a sparse linear system.

By the end, we will have implemented and checked all four user-model operations:

1. `sample_randomness()`
2. `couple_inputs()`
3. `build_linear_system()`
4. `quantity_of_interest()`

## 1. What problem are we solving?

At every level $\ell$, we solve the random linear system

$$
A_\ell u_\ell(X)=b_\ell(X),
$$

where $X\sim N(0,1)$ is a scalar random variable and $u_\ell(X)$ is the random solution vector at level $\ell$. The limiting random quantity we want to approximate is

$$
Q(X)=X.
$$

Because this is a manufactured test problem, we choose a known finite-level approximation. Define

$$
q_\ell(X) = X+h_\ell\sqrt{10^{-4}+|X|}.
$$

Here, $X$ is the limiting scalar quantity and

$$
e_\ell(X) = h_\ell\sqrt{10^{-4}+|X|}
$$

is the level-dependent approximation error. To ensure the approximation error decreases as the level resolution
increases, we define the level resolution as

$$
h_\ell=\frac{1}{n_\ell},
$$

where $n_\ell$ is the number of unknowns at level $\ell$:

$$
n_\ell=4\,2^\ell.
$$

The number of unknowns doubles at every level, so $h_\ell$ is halved:

$$
n_0=4,\qquad n_1=8,\qquad n_2=16, \quad \ldots
$$

$$
h_0=\frac14,\qquad h_1=\frac18,\qquad h_2=\frac1{16}, \quad \ldots
$$

Consequently, as $\ell\to\infty$

$$
h_\ell \to 0
$$

and

$$
q_\ell(X)\to X
$$

The desired, known solution of the level-$\ell$ system is the uniform vector

$$
u_\ell^*(X)
=
q_\ell(X)\mathbf{1}_{n_\ell},
$$

<!-- where $\mathbf{1}_{n_\ell}$ is a vector of $n_\ell$ ones. -->

<br>
<!-- 
We are also free to choose $A_\ell$, so we choose the sparse, symmetric positive-definite matrix

$$
A_\ell = (\ell+2)I_{n_\ell}
$$

and construct the right-hand side from the known level solution:

$$
b_\ell(X) = A_\ell u_\ell^*(X).
$$

Therefore, solving

$$
A_\ell u_\ell(X)=b_\ell(X)
$$

should recover

$$
u_\ell(X)=u_\ell^*(X).
$$

As the level increases,

$$
Q_\ell(X)\to Q(X)=X,
$$

and the vector solution approaches the corresponding uniform limiting solution:

$$
u_\ell(X)\to X \, \mathbf{1}_{n_\ell}.
$$ -->



In [8]:
from dataclasses import dataclass

import numpy as np
from scipy import sparse

from mlmc_linear_systems.linear_solver import (
    LinearSystem,
    direct_solve,
)
from mlmc_linear_systems.mlmc_model import CoupledInputs, MultilevelModel

## 2. How are $A_\ell$ and $b_\ell$ determined?

This is a **manufactured-solution test**. We first choose the known
level-dependent solution

$$
u_\ell^*(X) = q_\ell(X)\mathbf{1}_{n_\ell},
$$

where $\mathbf{1}_{n_\ell}$ is a vector of $n_\ell$ ones. We then choose a simple matrix $A_\ell$ and construct the right-hand side so
that $u_\ell^*(X)$ is the exact solution of the level system.

For this example, choose the sparse, symmetric positive-definite matrix

$$
A_\ell = (\ell+2)I_{n_\ell},
$$

Construct the right-hand side from the known solution and substituting in the definitions gives:

$$
\begin{aligned}
b_\ell(X) &= A_\ell u_\ell^*(X) \\
          &= (\ell+2)I_{n_\ell} \left[ q_\ell(X)\mathbf{1}_{n_\ell} \right] \\
          &= (\ell+2)q_\ell(X)\mathbf{1}_{n_\ell}.
\end{aligned}
$$

The complete level system is therefore

$$
(\ell+2)I_{n_\ell}u_\ell(X)
=
(\ell+2)q_\ell(X)\mathbf{1}_{n_\ell}.
$$

Because $I_{n_\ell}$ is the identity matrix, solving this system gives

$$
u_\ell(X)
=
q_\ell(X)\mathbf{1}_{n_\ell}
=
u_\ell^*(X).
$$

Thus, the numerical solver should recover the known manufactured
solution exactly, up to floating-point error.

The matrix $A_\ell$ is not intended to represent a physical operator.
It was chosen because it is sparse, symmetric positive definite, and
easy to solve analytically. This lets us test the linear solver and the
MLMC workflow against a known answer.

## 3. Implement the user model
We will now implement the problem using the `MultilevelModel` interface defined in [`mlmc_model.py`](../src/mlmc_linear_systems/mlmc_model.py).

### 3.1 General model implementation
A user creates a model class that satisfies the `MultilevelModel` protocol. Every model will be different and may contain any additional attributes, helper methods, meshes, operators, or cached data it needs.

#### Generic types
The protocol uses two generic types:

- `RandomnessT`: the type of the raw random object drawn by `sample_randomness()` for one coupled correction;
- `ModelInputT`: the model-ready, level-specific information passed to `build_linear_system()` to construct one linear system at one level.

These types describe two different stages of the model workflow. They do not necessarily contain different numerical information.

For example, `RandomnessT` could be:

- a scalar random variable;
- a vector of independent standard-normal variables;
- a random matrix;
- a random field;
- a custom object containing several random quantities.

The corresponding `ModelInputT` could contain:

- material coefficients;
- a forcing vector;
- boundary data;
- a level-specific coefficient field;
- any other information required to assemble the linear system.

#### Required operations

Every user model must implement four methods:

| Method | General responsibility |
|---|---|
| `sample_randomness()` | Draw the raw randomness needed for one coupled correction |
| `couple_inputs()` | Convert the shared randomness into fine and coarse model inputs |
| `build_linear_system()` | Construct one level-specific `LinearSystem(A, b)` |
| `quantity_of_interest()` | Reduce one solved system to a scalar output |

The interface has the following structure:

```python
class MultilevelModel(Protocol[RandomnessT, ModelInputT]):
    def sample_randomness(
        self,
        fine_level: int,
        rng: np.random.Generator,
    ) -> RandomnessT:
        """Draw the raw randomness needed for one MLMC correction.

        The returned object must contain enough random information to
        construct an input at ``fine_level`` and, when ``fine_level > 0``,
        a coupled input at ``fine_level - 1``.

        The returned object does not need to be a completed model input.
        """
        ...

    def couple_inputs(
        self,
        fine_level: int,
        randomness: RandomnessT,
    ) -> CoupledInputs[ModelInputT]:
        """Construct model-ready fine and coarse inputs.

        ``randomness`` is the object returned by ``sample_randomness()`` for
        the same fine level.

        The construction is model-specific. It may copy a scalar, evaluate
        a KL expansion, transfer random data between discretizations, or
        solve auxiliary SPDEs.

        The fine and coarse inputs may have different dimensions or
        representations, but they must come from the same underlying random
        realization.

        At ``fine_level == 0``, the coarse input must be ``None``.
        """
        ...

    def build_linear_system(
        self,
        level: int,
        model_input: ModelInputT,
    ) -> LinearSystem:
        """Build one sample-dependent linear system at one level.

        The returned object contains the matrix, right-hand side, and
        optional initial guess representing

            A_level x_level = b_level.

        This method constructs only one physical system. It does not draw
        randomness, construct the adjacent-level input, solve the system, or
        form the MLMC correction.
        """
        ...

    def quantity_of_interest(
        self,
        level: int,
        solution: np.ndarray,
        model_input: ModelInputT,
    ) -> float:
        """Calculate the scalar quantity of interest for one solution.

        The solution must come from the system constructed using the same
        ``level`` and ``model_input``.

        The returned scalar is Q_level.
        """
        ...
```
The user model defines:

- how randomness is sampled;
- how fine and coarse inputs are coupled;
- how each level-dependent system is assembled;
- how the scalar quantity of interest is calculated.

The MLMC runner will call these methods, solve the fine and coarse systems, and form the corresponding MLMC correction.

### 3.2 Synthetic-model implementation

For the synthetic model, the concrete generic types are:

| Generic type | Concrete type | Meaning |
|---|---|---|
| `RandomnessT` | `float` | The sampled scalar $X\sim N(0,1)$ |
| `ModelInputT` | `SyntheticModelInput` | The input used to construct one level system |

The corresponding protocol specialization is:

```python
MultilevelModel[float, SyntheticModelInput]
```

#### `SyntheticModelInput`

The input for one synthetic level is represented by:

```python
@dataclass(frozen=True)
class SyntheticModelInput:
    """Input used to construct one level-specific synthetic system."""

    random_value: float
```

`SyntheticModelInput` contains the sampled value $X$ needed to calculate
$q_\ell(X)$ and construct the level-dependent right-hand side.

It represents the input for only one level. It does not contain:

- the level number;
- the fine and coarse inputs together;
- the matrix or right-hand side;
- the solution;
- the quantity of interest.

The level is supplied separately:

```python
model.build_linear_system(
    level,
    model_input,
)
```

For this simple example, `SyntheticModelInput` is only a wrapper around the raw
scalar $X$. This wrapper is not mathematically necessary, but it demonstrates
the distinction between raw randomness and a model-ready input.

In a more complicated model, the conversion from `RandomnessT` to
`ModelInputT` might construct coefficient fields, forcing data, or other
level-specific information.

#### Coupling the synthetic inputs

For a correction with `fine_level > 0`, `couple_inputs()` constructs two
`SyntheticModelInput` objects containing the same sampled value:

```python
CoupledInputs(
    fine=SyntheticModelInput(random_value=X),
    coarse=SyntheticModelInput(random_value=X),
)
```

These are separate model-input objects, but they correspond to the same random
realization $X$.

The same $X$ must be used at both levels. If the coarse input used an
independently sampled value, the fine and coarse quantities would not form a
valid coupled MLMC correction.

At level zero, there is no preceding level:

```python
CoupledInputs(
    fine=SyntheticModelInput(random_value=X),
    coarse=None,
)
```

#### `SyntheticLevel` and the stored hierarchy

`SyntheticModelInput` contains the random information for one sample, while `SyntheticLevel` contains deterministic information that can be constructed once and reused for every sample:

```python
@dataclass(frozen=True)
class SyntheticLevel:
    """Deterministic data stored for one synthetic level."""

    index: int
    size: int
    step: float
    matrix: sparse.csr_matrix
```

When `SyntheticLinearModel` is created, it receives `number_of_levels` and constructs a tuple containing all available levels. For example, `SyntheticLinearModel(number_of_levels=4)` constructs levels $0$, $1$, $2$, and $3$ before any randomness is sampled.

For each level, `_create_level()` calculates $n_\ell$, $h_\ell$, and $A_\ell$. These quantities do not depend on $X$, so they are stored and reused. The random right-hand side $b_\ell(X)$ is constructed later by `build_linear_system()` for each sample. This mirrors a finite-element model that prepares its meshes, spaces, and deterministic operators before the MLMC sampling begins.

#### `SyntheticLinearModel`

The synthetic model satisfies the protocol by implementing the four required
methods:

```python
class SyntheticLinearModel:
    ...
```

First, create the concrete model:

```python
model = SyntheticLinearModel(number_of_levels=4)
```

The model can then be assigned to a variable annotated with the protocol:

```python
mlmc_model: MultilevelModel[
    float,
    SyntheticModelInput,
] = model
```

This assignment does not create a new model, copy the model, or wrap it in another object. `model` and `mlmc_model` refer to the same `SyntheticLinearModel` instance. The annotation tells a static type checker to verify that `SyntheticLinearModel` satisfies `MultilevelModel[float, SyntheticModelInput]`.

The two type arguments specify the concrete protocol types:

- `float` replaces `RandomnessT`, so `sample_randomness()` must return a `float`;
- `SyntheticModelInput` replaces `ModelInputT`, so `couple_inputs()` must return `CoupledInputs[SyntheticModelInput]`, while `build_linear_system()` and `quantity_of_interest()` must accept `SyntheticModelInput`.

The future MLMC runner can accept `mlmc_model` and use only the four methods guaranteed by the protocol. Additional helpers such as `level_size()` remain available through the concrete `model` variable.

For this model, the required methods have the following responsibilities:

| Method | Synthetic-model responsibility |
|---|---|
| `sample_randomness()` | Draw the scalar $X\sim N(0,1)$ |
| `couple_inputs()` | Construct fine and coarse inputs containing the same $X$ |
| `build_linear_system()` | Retrieve the stored level and construct $b_\ell(X)$ |
| `quantity_of_interest()` | Return the mean of the solved vector |

The model also defines helper methods for constructing and accessing its stored hierarchy:

| Helper method | Responsibility |
|---|---|
| `_create_level()` | Construct and store the deterministic data for one level |
| `_get_level()` | Validate a level index and return the stored `SyntheticLevel` |
| `level_size()` | Return the stored value $n_\ell=4\,2^\ell$ |
| `level_step()` | Return the stored value $h_\ell=1/n_\ell$ |
| `level_value()` | Calculate $q_\ell(X)$ |

The method `level_value()` evaluates

$$
q_\ell(X)
=
X+h_\ell\sqrt{10^{-4}+|X|}.
$$

The method `_create_level()` constructs and stores

$$
A_\ell=(\ell+2)I_{n_\ell}
$$

before sampling begins. For each new random input, `build_linear_system()` reuses this matrix and constructs

$$
b_\ell(X)
=
(\ell+2)q_\ell(X)\mathbf{1}_{n_\ell}.
$$

Solving the system should therefore recover

$$
u_\ell(X)
=
q_\ell(X)\mathbf{1}_{n_\ell}.
$$

#### Quantity of interest

The quantity of interest is the mean of the solution vector:

$$
Q_\ell(X)
=
\frac{1}{n_\ell}
\mathbf{1}_{n_\ell}^{\mathsf T}u_\ell(X).
$$

Because every component of the manufactured solution equals $q_\ell(X)$,

$$
Q_\ell(X)=q_\ell(X).
$$

The `quantity_of_interest()` method therefore returns:

```python
float(np.mean(solution))
```

#### Execution order

For a correction with `fine_level > 0`, the model is evaluated as follows:

```text
sample_randomness(fine_level, rng)
        |
        | returns the raw scalar X
        v
couple_inputs(fine_level, X)
        |
        | constructs two inputs from the same X
        |
        +-------------------------------+
        |                               |
        v                               v
fine SyntheticModelInput              coarse SyntheticModelInput
        |                               |
        v                               v
build_linear_system(             build_linear_system(
    fine_level,                      fine_level - 1,
    fine_input,                      coarse_input,
)                                )
        |                               |
        v                               v
solve fine system               solve coarse system
        |                               |
        v                               v
calculate Q_fine(X)             calculate Q_coarse(X)
        |                               |
        +---------------+---------------+
                        |
                        v
             Y_level = Q_fine - Q_coarse
```

For $\ell>0$, the correction is

$$
\begin{aligned}
Y_\ell
&=Q_\ell(X)-Q_{\ell-1}(X)\\
&=q_\ell(X)-q_{\ell-1}(X)\\
&=(h_\ell-h_{\ell-1})
  \sqrt{10^{-4}+|X|}.
\end{aligned}
$$

The shared random term $X$ cancels because the same realization is used at
both levels.

At level zero, only the fine system is constructed and solved:

$$
Y_0=Q_0(X).
$$

The next code cell implements `SyntheticModelInput`, `SyntheticLevel`, and `SyntheticLinearModel`
using this structure.

In [9]:
@dataclass(frozen=True)
class SyntheticModelInput:
    """Random input used to construct one synthetic level system."""

    random_value: float


@dataclass(frozen=True)
class SyntheticLevel:
    """Deterministic data stored for one synthetic level."""

    index: int
    size: int
    step: float
    matrix: sparse.csr_matrix


class SyntheticLinearModel:
    """Manufactured hierarchy with preconstructed deterministic levels."""

    def __init__(self, number_of_levels: int):
        """Construct and store all available model levels."""
        if number_of_levels <= 0:
            raise ValueError("number_of_levels must be positive.")

        self.number_of_levels = number_of_levels
        self.levels: tuple[SyntheticLevel, ...] = tuple(
            self._create_level(level)
            for level in range(number_of_levels)
        )

    @staticmethod
    def _create_level(level: int) -> SyntheticLevel:
        """Construct the deterministic data for one level."""
        size = 4 * 2**level
        step = 1.0 / size

        matrix = sparse.eye(
            size,
            format="csr",
            dtype=float,
        ) * float(level + 2)

        return SyntheticLevel(
            index=level,
            size=size,
            step=step,
            matrix=matrix,
        )

    def _get_level(self, level: int) -> SyntheticLevel:
        """Return one stored level after validating its index."""
        if level < 0 or level >= self.number_of_levels:
            raise ValueError(
                f"level must be between 0 and "
                f"{self.number_of_levels - 1}."
            )

        return self.levels[level]

    def level_size(self, level: int) -> int:
        """Return the number of unknowns at one stored level."""
        return self._get_level(level).size

    def level_step(self, level: int) -> float:
        """Return the approximation step at one stored level."""
        return self._get_level(level).step

    def sample_randomness(
        self,
        fine_level: int,
        rng: np.random.Generator,
    ) -> float:
        """Draw the standard-normal scalar for one correction."""
        self._get_level(fine_level)
        return float(rng.normal())

    def couple_inputs(
        self,
        fine_level: int,
        randomness: float,
    ) -> CoupledInputs[SyntheticModelInput]:
        """Construct adjacent-level inputs from the same random value."""
        self._get_level(fine_level)

        fine_input = SyntheticModelInput(
            random_value=float(randomness),
        )

        if fine_level == 0:
            return CoupledInputs(
                fine=fine_input,
                coarse=None,
            )

        coarse_input = SyntheticModelInput(
            random_value=float(randomness),
        )

        return CoupledInputs(
            fine=fine_input,
            coarse=coarse_input,
        )

    def level_value(
        self,
        level: int,
        model_input: SyntheticModelInput,
    ) -> float:
        """Calculate the manufactured scalar value q_level(X)."""
        level_data = self._get_level(level)
        random_value = model_input.random_value

        level_error = level_data.step * np.sqrt(
            1e-4 + abs(random_value)
        )

        return float(random_value + level_error)

    def build_linear_system(
        self,
        level: int,
        model_input: SyntheticModelInput,
    ) -> LinearSystem:
        """Inject the random input into one preconstructed level."""
        level_data = self._get_level(level)
        level_value = self.level_value(level, model_input)

        expected_solution = np.full(
            level_data.size,
            level_value,
        )

        right_hand_side = np.asarray(
            level_data.matrix @ expected_solution
        )

        return LinearSystem(
            A=level_data.matrix,
            b=right_hand_side,
        )

    def quantity_of_interest(
        self,
        level: int,
        solution: np.ndarray,
        model_input: SyntheticModelInput,
    ) -> float:
        """Return the mean solution as the level quantity of interest."""
        level_data = self._get_level(level)
        solution_vector = np.asarray(solution)

        expected_shape = (level_data.size,)

        if solution_vector.shape != expected_shape:
            raise ValueError(
                f"Solution for level {level} must have shape "
                f"{expected_shape}."
            )

        return float(np.mean(solution_vector))

## 4. Create the model and choose the levels

The synthetic model follows the same setup pattern as a model with a more expensive discretization: it receives the number of levels when it is created and stores the deterministic data for those levels before any random samples are drawn. We will use

```python
number_of_levels = 4
```

which creates levels $0$, $1$, $2$, and $3$. Constructing these synthetic levels is inexpensive, but it mirrors how an NGSolve model would prepare its meshes, finite-element spaces, forms, and other deterministic level data before the MLMC sampling begins.

The random-number generator is created with a fixed seed so that rerunning the notebook produces the same samples. `model` retains the concrete `SyntheticLinearModel` type, while `mlmc_model` provides the protocol-typed view that a future runner will use. Both names refer to the same model object.

In [10]:
rng = np.random.default_rng(12345)
number_of_levels = 4


model = SyntheticLinearModel(number_of_levels=number_of_levels)

mlmc_model: MultilevelModel[
    float,
    SyntheticModelInput,
] = model

### Inspect the stored hierarchy

Before drawing any randomness, we inspect the deterministic data stored at each level. The output shows that the number of unknowns doubles from one level to the next, while the level step is halved. Each sparse matrix has the dimensions required by its level and will be reused for every random sample evaluated at that level.

In [11]:
for level_data in model.levels:
    print(
        f"level={level_data.index}, "
        f"size={level_data.size}, "
        f"step={level_data.step:.5f}, "
        f"matrix_shape={level_data.matrix.shape}"
    )

level=0, size=4, step=0.25000, matrix_shape=(4, 4)
level=1, size=8, step=0.12500, matrix_shape=(8, 8)
level=2, size=16, step=0.06250, matrix_shape=(16, 16)
level=3, size=32, step=0.03125, matrix_shape=(32, 32)


## 5. Construct one coupled sample

We will manually evaluate the correction whose fine level is $\ell=2$:

$$
Y_2=Q_2-Q_1.
$$

The future MLMC runner will automate these steps, but here we perform them
explicitly to demonstrate the model interface.

In [12]:
fine_level = 2

randomness = mlmc_model.sample_randomness(
    fine_level,
    rng,
)

coupled_inputs = mlmc_model.couple_inputs(
    fine_level,
    randomness,
)

print("randomness:", randomness)
print("fine input:", coupled_inputs.fine)
print("coarse input:", coupled_inputs.coarse)

randomness: -1.4238250364546312
fine input: SyntheticModelInput(random_value=-1.4238250364546312)
coarse input: SyntheticModelInput(random_value=-1.4238250364546312)


The printed inputs are two separate `SyntheticModelInput` objects, but both contain the same sampled value $X$. The first assertion confirms that a coarse input exists for level $2$, and the second confirms that the fine and coarse systems will be driven by the same underlying randomness rather than independent samples.

In [13]:
assert coupled_inputs.coarse is not None
assert (
    coupled_inputs.fine.random_value
    == coupled_inputs.coarse.random_value
)

## 6. Build the fine and coarse systems

The requested correction has fine level $2$, so its coarse level is $1$. We pass each level its matching `SyntheticModelInput`. `build_linear_system()` retrieves the matrix stored during model construction and creates the sample-dependent right-hand side using the shared value $X$. The resulting systems have different dimensions because the two levels have different numbers of unknowns.

In [14]:
fine_system = mlmc_model.build_linear_system(
    fine_level,
    coupled_inputs.fine,
)

coarse_level = fine_level - 1

coarse_system = mlmc_model.build_linear_system(
    coarse_level,
    coupled_inputs.coarse,
)

print("fine matrix shape:", fine_system.A.shape)
print("coarse matrix shape:", coarse_system.A.shape)

fine matrix shape: (16, 16)
coarse matrix shape: (8, 8)


The following assertions use Python's `is` operator to check object identity, not merely equality of matrix entries. They verify that `build_linear_system()` returns the same matrix objects that were prepared when the model hierarchy was created. Only the right-hand sides depend on the current random sample.

In [15]:
assert fine_system.A is model.levels[fine_level].matrix
assert coarse_system.A is model.levels[coarse_level].matrix

## 7. Solve the systems and form the correction

The fine and coarse systems are solved independently after their inputs have been coupled. Each solution is then reduced to a scalar quantity of interest by taking its mean. The correction for this sample is

$$
Y_2=Q_2-Q_1.
$$

The solver-success assertions ensure that a failed linear solve cannot silently contribute an invalid value to the correction.

In [16]:
fine_result = direct_solve(
    fine_system.A,
    fine_system.b,
)

coarse_result = direct_solve(
    coarse_system.A,
    coarse_system.b,
)

assert fine_result.success
assert coarse_result.success

fine_qoi = mlmc_model.quantity_of_interest(
    fine_level,
    fine_result.solution,
    coupled_inputs.fine,
)

coarse_qoi = mlmc_model.quantity_of_interest(
    coarse_level,
    coarse_result.solution,
    coupled_inputs.coarse,
)

correction = fine_qoi - coarse_qoi

print("fine QoI:", fine_qoi)
print("coarse QoI:", coarse_qoi)
print("correction:", correction)

fine QoI: -1.349244830141891
coarse QoI: -1.2746646238291506
correction: -0.07458020631274032


## 8. Check the correction against the analytical result

Because the same $X$ is used at both levels, the shared random term cancels and the correction is known exactly:

$$
Y_\ell=(h_\ell-h_{\ell-1})\sqrt{10^{-4}+|X|}.
$$

For this hierarchy, $h_2<h_1$, so the correction is negative. The numerical and analytical values may differ by a very small floating-point rounding error, which is why the comparison uses `np.isclose()` instead of exact equality.

In [17]:
expected_correction = (
    model.level_step(fine_level)
    - model.level_step(coarse_level)
) * np.sqrt(
    1e-4 + abs(randomness)
)

print("expected correction:", expected_correction)

assert np.isclose(
    correction,
    expected_correction,
)

expected correction: -0.07458020631274026


## 9. Evaluate the level-zero correction

Level zero is the base of the telescoping MLMC estimator and has no preceding coarse level. Therefore, `couple_inputs()` must return `coarse=None`, only one linear system is solved, and the correction is simply

$$
Y_0=Q_0.
$$

This is the special branch that a future runner will use whenever it samples level zero.

In [18]:
level_zero_randomness = mlmc_model.sample_randomness(
    0,
    rng,
)

level_zero_inputs = mlmc_model.couple_inputs(
    0,
    level_zero_randomness,
)

assert level_zero_inputs.coarse is None

level_zero_system = mlmc_model.build_linear_system(
    0,
    level_zero_inputs.fine,
)

level_zero_result = direct_solve(
    level_zero_system.A,
    level_zero_system.b,
)

level_zero_qoi = mlmc_model.quantity_of_interest(
    0,
    level_zero_result.solution,
    level_zero_inputs.fine,
)

print("Y_0 =", level_zero_qoi)

Y_0 = 1.5447787714772179


## 10. What this example verified

This notebook manually performed the work that a future MLMC runner will automate. It verified that:

- deterministic level data are constructed once and reused across samples;
- `sample_randomness()` draws the raw randomness for one correction;
- `couple_inputs()` gives adjacent levels inputs derived from the same random realization;
- `build_linear_system()` combines stored level data with the current random input;
- the fine and coarse systems are solved independently;
- `quantity_of_interest()` reduces each solution to a scalar;
- the numerical correction agrees with the known analytical correction;
- level zero correctly uses $Y_0=Q_0$ with no coarse solve.

The next package component can reuse this exact sequence to evaluate one correction programmatically before it is placed inside a repeated-sampling MLMC runner.